# Phase 5.4 — Investigation Packets (SOC Triage Outputs)

Phase 5.3 produced a small, gated alert queue (SOC-ready volume).
Phase 5.4 converts each alert into an **investigation packet**:

- why fired (signal conditions)
- top contributing telemetry features (rz scores)
- baseline deviation summary
- context fields (entity/service/source_file)
- suggested analyst actions
- confidence + limitations

This turns detection engineering into analyst-ready operational output.


## 1) Setup & Output Directories

We save artifacts to:

- `outputs/alerts/` (JSONL + CSV packets)
- `outputs/tables/` (summary tables)


In [1]:
import os
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

PHASE5_DIR = r"D:\Projects\CICIDS-2017\03-security-integration\cicids-2017\phase-5"
OUTPUT_DIR = os.path.join(PHASE5_DIR, "outputs")

TAB_DIR = os.path.join(OUTPUT_DIR, "tables")
ALERT_DIR = os.path.join(OUTPUT_DIR, "alerts")

os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(ALERT_DIR, exist_ok=True)

print("TAB_DIR:", TAB_DIR)
print("ALERT_DIR:", ALERT_DIR)


TAB_DIR: D:\Projects\CICIDS-2017\03-security-integration\cicids-2017\phase-5\outputs\tables
ALERT_DIR: D:\Projects\CICIDS-2017\03-security-integration\cicids-2017\phase-5\outputs\alerts


## 2) Load final alerts from Phase 5.3

Phase 5.3 saved the gated alerts table:
`outputs/tables/5.3_final_alerts_after_gating.csv`

We load it as input and generate investigation packets.


In [2]:
alerts_path = os.path.join(TAB_DIR, "5.3_final_alerts_after_gating.csv")
alerts = pd.read_csv(alerts_path, low_memory=False)

print("Loaded gated alerts:", alerts.shape)
alerts.head()


Loaded gated alerts: (8, 89)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file,rz_Flow Duration,rz_Total Fwd Packets,rz_Total Backward Packets,rz_Flow Bytes/s,rz_Flow Packets/s,anomaly_score,signal_candidate,group_key,severity
0,60210,1,2,0,2071,0,2065,6,1035.5,1455.932862,0,0,0.0,0.0,2.071000e+09,2000000.0,1.0,0.0,1,1,1,1.0,0.0,1,1,0,0.0,0.0,0,0,1,0,0,0,40,0,2000000.0,0.0,6,2065,1378.666667,1188.764204,1.413160e+06,0,1,0,0,1,0,0,0,0,2068.0,1035.5,0.0,40,0,0,0,0,0,0,2,2071,0,0,16425,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv,1.001551,0.0,2.0,402152.683322,16373.958701,402152.683322,True,Monday-WorkingHours.pcap_ISCX.csv,Critical
1,16398,1,2,0,2065,0,2065,0,1032.5,1460.175503,0,0,0.0,0.0,2.070000e+09,2000000.0,1.0,0.0,1,1,1,1.0,0.0,1,1,0,0.0,0.0,0,0,1,0,0,0,40,0,2000000.0,0.0,0,2065,1376.666667,1192.228306,1.421408e+06,0,1,0,0,1,0,0,0,0,2065.0,1032.5,0.0,40,0,0,0,0,0,0,2,2065,0,0,16425,-1,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Tuesday-WorkingHours.pcap_ISCX.csv,1.001551,0.0,2.0,401958.499989,16373.958701,401958.499989,True,Tuesday-WorkingHours.pcap_ISCX.csv,High
2,45926,1,2,0,2071,0,2065,6,1035.5,1455.932862,0,0,0.0,0.0,2.070000e+09,2000000.0,1.0,0.0,1,1,1,1.0,0.0,1,1,0,0.0,0.0,0,0,1,0,0,0,40,0,2000000.0,0.0,6,2065,1378.666667,1188.764204,1.413160e+06,0,1,0,0,1,0,0,0,0,2068.0,1035.5,0.0,40,0,0,0,0,0,0,2,2071,0,0,16425,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Wednesday-workingHours.pcap_ISCX.csv,1.001551,0.0,2.0,401958.499989,16373.958701,401958.499989,True,Wednesday-workingHours.pcap_ISCX.csv,High
3,56792,1,2,0,2071,0,2065,6,1035.5,1455.932862,0,0,0.0,0.0,2.070000e+09,2000000.0,1.0,0.0,1,1,1,1.0,0.0,1,1,0,0.0,0.0,0,0,1,0,0,0,40,0,2000000.0,0.0,6,2065,1378.666667,1188.764204,1.413160e+06,0,1,0,0,1,0,0,0,0,2068.0,1035.5,0.0,40,0,0,0,0,0,0,2,2071,0,0,256,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,1.001551,0.0,2.0,401958.499989,16373.958701,401958.499989,True,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,High
4,7350,1,2,0,2071,0,2065,6,1035.5,1455.932862,0,0,0.0,0.0,2.070000e+09,2000000.0,1.0,0.0,1,1,1,1.0,0.0,1,1,0,0.0,0.0,0,0,1,0,0,0,40,0,2000000.0,0.0,6,2065,1378.666667,1188.764204,1.413160e+06,0,1,0,0,1,0,0,0,0,2068.0,1035.5,0.0,40,0,0,0,0,0,0,2,2071,0,0,256,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,Friday-WorkingHours-Morning.pcap_ISCX.csv,1.001551,0.0,2.0,401958.499989,16373.958701,401958.499989,True,Friday-WorkingHours-Morning.pcap_ISCX.csv,High


## 3) Identify required fields (entity, service, protocol, label, source_file)

We detect which context fields exist in CICIDS schema.
This keeps the notebook robust across environments.


In [3]:
def pick_col(candidates, cols):
    for c in candidates:
        if c in cols:
            return c
    return None

cols = set(alerts.columns)

label_col = pick_col(["Label","label","Class","class"], cols)
src_col = pick_col(["Src IP","Source IP","src_ip"], cols)
dstip_col = pick_col(["Dst IP","Destination IP","dst_ip"], cols)
dstport_col = pick_col(["Dst Port","Destination Port","dst_port"], cols)
proto_col = pick_col(["Protocol","protocol"], cols)

print("label_col:", label_col)
print("src_col:", src_col)
print("dstip_col:", dstip_col)
print("dstport_col:", dstport_col)
print("proto_col:", proto_col)


label_col: Label
src_col: None
dstip_col: None
dstport_col: Destination Port
proto_col: None


## 4) Feature deviation signals (rz_*) and top contributing features

Phase 5 anomaly scoring used robust z-scores per feature (`rz_<feature>`).

For each alert we extract:
- top 3 rz features
- their deviation values
This becomes part of the "why fired" explanation.


In [4]:
rz_cols = [c for c in alerts.columns if c.startswith("rz_")]
rz_cols[:10], len(rz_cols)


(['rz_Flow Duration',
  'rz_Total Fwd Packets',
  'rz_Total Backward Packets',
  'rz_Flow Bytes/s',
  'rz_Flow Packets/s'],
 5)

In [5]:
def top_contributors(row, rz_columns, k=3):
    pairs = []
    for c in rz_columns:
        val = row.get(c, np.nan)
        if pd.notna(val):
            pairs.append((c.replace("rz_", ""), float(val)))
    pairs.sort(key=lambda x: x[1], reverse=True)
    return pairs[:k]


## 5) Investigation Packet Template

Each alert becomes a structured object containing:

- alert metadata
- why fired (threshold + contributors)
- context fields
- recommended analyst actions
- confidence + known limitations

This mirrors real SOC workflows.


In [7]:
GENERIC_ACTIONS = [
    "Validate whether activity matches entity baseline (normal behavior range).",
    "Check repeated patterns across ports/services (scan-style behavior).",
    "Correlate with peer entities in same subnet/zone if available.",
    "Escalate if repeated anomalies persist or cluster around sensitive services.",
]

ATTACK_HINT_ACTIONS = [
    "If Dst Port is unusual or sequential across alerts, treat as possible scan.",
    "If traffic volume spikes abruptly, verify potential DDoS/abuse indicators.",
    "Check if anomaly is isolated burst or sustained behavior (persistence).",
]

In [8]:
def confidence_from_score(score):
    # simple heuristic: higher anomaly score => higher confidence
    if score >= 20:
        return "High"
    elif score >= 10:
        return "Medium"
    else:
        return "Low"


In [9]:
def build_packet(row, threshold_name="p995", threshold_value=None):
    packet = {}

    # core
    packet["alert_id"] = f"cicids-5.4-{int(row.name)}"
    packet["signal_type"] = "MAD_Robust_Anomaly"
    packet["severity"] = row.get("severity", "Unknown")

    # why fired
    anomaly_score = float(row.get("anomaly_score", np.nan))
    packet["why_fired"] = {
        "anomaly_score": anomaly_score,
        "threshold_name": threshold_name,
        "threshold_value": float(threshold_value) if threshold_value is not None else None,
        "top_contributors": top_contributors(row, rz_cols, k=3)
    }

    # context
    ctx = {
        "source_file": row.get("source_file", None),
        "label": row.get(label_col, None) if label_col else None
    }
    if src_col: ctx["src_ip"] = row.get(src_col, None)
    if dstip_col: ctx["dst_ip"] = row.get(dstip_col, None)
    if dstport_col: ctx["dst_port"] = row.get(dstport_col, None)
    if proto_col: ctx["protocol"] = row.get(proto_col, None)

    packet["context"] = ctx

    # actions
    packet["recommended_actions"] = GENERIC_ACTIONS + ATTACK_HINT_ACTIONS

    # confidence + limitations
    packet["confidence"] = confidence_from_score(anomaly_score)
    packet["limitations"] = [
        "CICIDS is simulated traffic; behavior may differ in production.",
        "Context fields (asset criticality, identity baselines) are limited.",
        "Anomaly score indicates deviation, not confirmed maliciousness."
    ]

    return packet


## 6) Generate investigation packets for all gated alerts

We create:
- JSONL file (one packet per line)
- CSV summary table (for quick browsing)


In [10]:
# If you used p995 gating in Phase 5.3:
threshold_name = "p995"

# optional: store threshold value manually if you want (not required)
threshold_value = None


In [11]:
packets = []
for idx, row in alerts.iterrows():
    row.name = idx
    packets.append(build_packet(row, threshold_name=threshold_name, threshold_value=threshold_value))

len(packets), packets[0]


(8,
 {'alert_id': 'cicids-5.4-0',
  'signal_type': 'MAD_Robust_Anomaly',
  'severity': 'Critical',
  'why_fired': {'anomaly_score': 402152.6833220217,
   'threshold_name': 'p995',
   'threshold_value': None,
   'top_contributors': [('Flow Bytes/s', 402152.6833220217),
    ('Flow Packets/s', 16373.958700632218),
    ('Total Backward Packets', 1.999999998)]},
  'context': {'source_file': 'Monday-WorkingHours.pcap_ISCX.csv',
   'label': 'BENIGN',
   'dst_port': 60210},
  'recommended_actions': ['Validate whether activity matches entity baseline (normal behavior range).',
   'Check repeated patterns across ports/services (scan-style behavior).',
   'Correlate with peer entities in same subnet/zone if available.',
   'Escalate if repeated anomalies persist or cluster around sensitive services.',
   'If Dst Port is unusual or sequential across alerts, treat as possible scan.',
   'If traffic volume spikes abruptly, verify potential DDoS/abuse indicators.',
   'Check if anomaly is isolated bu

In [12]:
jsonl_path = os.path.join(ALERT_DIR, "5.4_investigation_packets.jsonl")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for p in packets:
        f.write(json.dumps(p) + "\n")

print("Saved:", jsonl_path)


Saved: D:\Projects\CICIDS-2017\03-security-integration\cicids-2017\phase-5\outputs\alerts\5.4_investigation_packets.jsonl


In [13]:
summary_rows = []
for p in packets:
    tc = p["why_fired"]["top_contributors"]
    top1 = tc[0][0] if len(tc) > 0 else None
    top1_val = tc[0][1] if len(tc) > 0 else None

    summary_rows.append({
        "alert_id": p["alert_id"],
        "severity": p["severity"],
        "anomaly_score": p["why_fired"]["anomaly_score"],
        "top_feature": top1,
        "top_feature_rz": top1_val,
        "source_file": p["context"].get("source_file"),
        "label": p["context"].get("label"),
        "src_ip": p["context"].get("src_ip") if "src_ip" in p["context"] else None,
        "dst_port": p["context"].get("dst_port") if "dst_port" in p["context"] else None,
    })

packets_df = pd.DataFrame(summary_rows)
packets_df


,alert_id,severity,anomaly_score,top_feature,top_feature_rz,source_file,label,src_ip,dst_port
0,cicids-5.4-0,Critical,402152.683322,Flow Bytes/s,402152.683322,Monday-WorkingHours.pcap_ISCX.csv,BENIGN,None,60210
1,cicids-5.4-1,High,401958.499989,Flow Bytes/s,401958.499989,Tuesday-WorkingHours.pcap_ISCX.csv,BENIGN,None,16398
2,cicids-5.4-2,High,401958.499989,Flow Bytes/s,401958.499989,Wednesday-workingHours.pcap_ISCX.csv,BENIGN,None,45926
3,cicids-5.4-3,High,401958.499989,Flow Bytes/s,401958.499989,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,BENIGN,None,56792
4,cicids-5.4-4,High,401958.499989,Flow Bytes/s,401958.499989,Friday-WorkingHours-Morning.pcap_ISCX.csv,BENIGN,None,7350
5,cicids-5.4-5,High,401958.499989,Flow Bytes/s,401958.499989,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,BENIGN,None,21161
6,cicids-5.4-6,High,401958.499989,Flow Bytes/s,401958.499989,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,BENIGN,None,6830
7,cicids-5.4-7,High,401958.499989,Flow Bytes/s,401958.499989,Thursday-WorkingHours-Afternoon-Infilteration....,BENIGN,None,17101


In [14]:
csv_path = os.path.join(ALERT_DIR, "5.4_investigation_packets.csv")
packets_df.to_csv(csv_path, index=False)
print("Saved:", csv_path)


Saved: D:\Projects\CICIDS-2017\03-security-integration\cicids-2017\phase-5\outputs\alerts\5.4_investigation_packets.csv


## 7) Investigation packet quality checks

We sanity check:
- severity distribution
- label distribution
- common top contributors

This ensures packets are interpretable and SOC-friendly.


In [15]:
packets_df["severity"].value_counts(), packets_df["label"].value_counts()


(severity
 High        7
 Critical    1
 Name: count, dtype: int64,
 label
 BENIGN    8
 Name: count, dtype: int64)

In [16]:
packets_df["top_feature"].value_counts().head(10)


top_feature
Flow Bytes/s    8
Name: count, dtype: int64

## Takeaways (Phase 5.4)

- Alerts are now analyst-actionable via investigation packets.
- Each alert explains: why fired + which features drove deviation.
- Context fields are included where dataset supports them.
- Recommended actions make the output SOC-triage aligned.

✅ Next: Phase 5.5 SOC Evaluation Metrics (workload, FP/day, stability).
